# Model Inference


## Imports


In [ ]:
import os
import sys
from pathlib import Path
import warnings
import zipfile

warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import mlflow
import mlflow.pyfunc
from mlflow import MlflowClient
import dagshub

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.features import load_raw_data, merge_all

DATA_DIR = PROJECT_ROOT
print('Imports ready')


Imports ready


## MLflow Setup


In [ ]:
os.environ.setdefault('MLFLOW_TRACKING_USERNAME', 'sansi23')

dagshub.init(
    repo_owner='sansi23',
    repo_name='Walmart-Recruiting---Store-Sales-Forecasting',
    mlflow=True
)

print('Tracking URI:', mlflow.get_tracking_uri())


Initialized MLflow to track repo "sansi23/Walmart-Recruiting---Store-Sales-Forecasting"

Repository sansi23/Walmart-Recruiting---Store-Sales-Forecasting initialized!

Tracking URI: https://dagshub.com/sansi23/Walmart-Recruiting---Store-Sales-Forecasting.mlflow


## Load Test Data


In [ ]:
train_raw, test_raw, features_raw, stores_raw = load_raw_data(DATA_DIR)
_, test_raw = merge_all(
    train_raw,
    test_raw,
    features_raw,
    stores_raw,
)

test_raw = test_raw.reset_index(drop=True)
prediction_ids = (
    test_raw['Store'].astype(str) + '_' +
    test_raw['Dept'].astype(str) + '_' +
    pd.to_datetime(test_raw['Date']).dt.strftime('%Y-%m-%d')
)

assert not prediction_ids.duplicated().any()

print('Test shape:', test_raw.shape)
print('Test dates:', test_raw['Date'].min(), '->', test_raw['Date'].max())


Test shape: (115064, 15)
Test dates: 2012-11-02 00:00:00 -> 2013-07-26 00:00:00


## Load Best CatBoost Pipeline


In [ ]:
MODEL_NAME = 'WalmartSales_CatBoost_RawDepth10'
client = MlflowClient()
versions = list(client.search_model_versions(f"name='{MODEL_NAME}'"))

if not versions:
    raise RuntimeError(f'{MODEL_NAME} is not registered. Run its final pipeline cell first.')

valid_versions = []
for version in versions:
    if not version.run_id:
        continue
    params = client.get_run(version.run_id).data.params
    if params.get('depth') == '10' and params.get('training_feature_contract') == 'origin_style':
        valid_versions.append(version)

if not valid_versions:
    raise RuntimeError('No registered depth-10 origin-style CatBoost version was found. Run the corrected final training cell.')

latest_version = max(valid_versions, key=lambda item: int(item.version))
MODEL_VERSION = int(latest_version.version)
MODEL_URI = f'models:/{MODEL_NAME}/{MODEL_VERSION}'

pipeline = mlflow.pyfunc.load_model(MODEL_URI)
sklearn_pipeline = pipeline._model_impl.sklearn_model
sklearn_pipeline.fitted_ = True
if 'catboost' in sklearn_pipeline.named_steps:
    sklearn_pipeline.named_steps['catboost'].fitted_ = True
feature_step = sklearn_pipeline.named_steps.get('features')
feature_columns = list(getattr(feature_step, 'feature_cols_', []))
forbidden_features = {
    'Sales_Lag1', 'Sales_Lag2', 'Sales_Lag4', 'Sales_Lag13', 'Sales_Lag26',
    'Sales_Roll_4w_Mean', 'Sales_Roll_13w_Mean', 'Sales_Roll_26w_Mean',
    'Sales_Roll_52w_Mean',
}
unexpected_features = forbidden_features.intersection(feature_columns)
if unexpected_features or 'Sales_Lag52_Origin' not in feature_columns:
    raise RuntimeError(f'Loaded model has an invalid CatBoost feature contract: {feature_columns}')

print('Selected model: CatBoost RawDepth10')
print('Registered model:', MODEL_NAME)
print('Version:', MODEL_VERSION)


AttributeError: Can't get attribute 'build_origin_training_frame' on <module 'src.features' from 'c:\\Users\\user\\Desktop\\ML Final Project\\src\\features.py'>

## Predict


In [ ]:
prediction_output = pipeline.predict(test_raw)

if isinstance(prediction_output, pd.DataFrame):
    if 'Weekly_Sales' in prediction_output.columns:
        test_predictions = prediction_output['Weekly_Sales'].to_numpy()
    elif prediction_output.shape[1] == 1:
        test_predictions = prediction_output.iloc[:, 0].to_numpy()
    else:
        raise ValueError(f'Unsupported prediction columns: {prediction_output.columns.tolist()}')
elif isinstance(prediction_output, pd.Series):
    test_predictions = prediction_output.to_numpy()
else:
    test_predictions = np.asarray(prediction_output).reshape(-1)

test_predictions = np.clip(test_predictions.astype(float), 0, None)

assert len(test_predictions) == len(test_raw)
assert np.isfinite(test_predictions).all()

prediction_frame = pd.DataFrame({
    'Id': prediction_ids,
    'Weekly_Sales': test_predictions,
})

print('Prediction rows:', len(prediction_frame))
print(prediction_frame['Weekly_Sales'].describe())
display(prediction_frame.head(10))


## Save Submission


In [ ]:
sample_path = os.path.join(DATA_DIR, 'sampleSubmission.csv')

if os.path.exists(sample_path):
    sample = pd.read_csv(sample_path)
else:
    with zipfile.ZipFile(os.path.join(DATA_DIR, 'sampleSubmission.csv.zip'), 'r') as archive:
        sample = pd.read_csv(archive.open(archive.namelist()[0]))

if set(prediction_frame['Id']) != set(sample['Id']):
    missing = sorted(set(sample['Id']) - set(prediction_frame['Id']))[:5]
    extra = sorted(set(prediction_frame['Id']) - set(sample['Id']))[:5]
    raise ValueError(f'ID mismatch. Missing={missing}; extra={extra}')

submission = sample[['Id']].merge(
    prediction_frame,
    on='Id',
    how='left',
)

assert len(submission) == len(sample)
assert submission['Id'].equals(sample['Id'])
assert submission['Weekly_Sales'].notna().all()
assert (submission['Weekly_Sales'] >= 0).all()

SUBMISSION_PATH = 'submission_catboost_pipeline.csv'
submission.to_csv(SUBMISSION_PATH, index=False)

print('Saved:', SUBMISSION_PATH)
display(submission.head(10))
